# Hybrid Model Selection

This notebook compares final recommender candidates after finding that `overview_genre_text` is the strongest content representation. The main question is whether an explicit genre score is still useful once genre information is already included in the content representation.

## Setup

Load the train/test split, MovieLens metadata, TMDb metadata, and recommender functions.

In [1]:
from pathlib import Path
import sys

current_path = Path.cwd().resolve()

for path in [current_path, *current_path.parents]:
    if (path / "src" / "data.py").exists():
        project_root = path
        break
else:
    raise FileNotFoundError("Could not find project root containing src/data.py")

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd

from src.content import build_movie_content_features
from src.data import load_movies
from src.metrics import precision_at_k, recall_at_k, ndcg_at_k, rated_precision_at_k
from src.recommenders import (
    build_content_tfidf_matrix,
    recommend_popular_movies,
    recommend_popularity_content_hybrid_movies_from_vectors,
    recommend_true_hybrid_movies_from_vectors,
)

train = pd.read_csv(project_root / "data/processed/train_ratings.csv")
test = pd.read_csv(project_root / "data/processed/test_ratings.csv")
movies = load_movies()
tmdb_metadata = pd.read_csv(project_root / "data/processed/tmdb_movie_metadata.csv")

movie_content = build_movie_content_features(
    movies=movies,
    tmdb_metadata=tmdb_metadata,
)

_, content_vectors = build_content_tfidf_matrix(
    movie_content,
    text_column="overview_genre_text",
)

movie_content[["movieId", "title", "overview_genre_text"]].head()

,movieId,title,overview_genre_text
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ..."
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom..."
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...


## Evaluation Helpers

Evaluate accuracy metrics, rated-only precision, and catalog breadth for each candidate model.

In [2]:
def evaluate_recommender(recommender_function, model_name: str, k: int = 10) -> dict:
    user_scores = []
    recommendation_rows = []

    positive_counts = (
        train[train["is_positive"]]
        .groupby("movieId")
        .size()
        .reset_index(name="global_positive_ratings")
    )

    for user_id in test["userId"].unique():
        recs = recommender_function(user_id=user_id, k=k).copy()

        recommended_items = recs["movieId"].tolist()

        user_test = test[test["userId"] == user_id]
        rated_items = set(user_test["movieId"])
        positive_items = set(user_test[user_test["is_positive"]]["movieId"])

        user_scores.append(
            {
                "precision_at_10": precision_at_k(
                    recommended_items=recommended_items,
                    relevant_items=positive_items,
                    k=k,
                ),
                "recall_at_10": recall_at_k(
                    recommended_items=recommended_items,
                    relevant_items=positive_items,
                    k=k,
                ),
                "ndcg_at_10": ndcg_at_k(
                    recommended_items=recommended_items,
                    relevant_items=positive_items,
                    k=k,
                ),
                "rated_precision_at_10": rated_precision_at_k(
                    recommended_items=recommended_items,
                    positive_items=positive_items,
                    rated_items=rated_items,
                    k=k,
                ),
            }
        )

        recs["userId"] = user_id
        recs["rank"] = range(1, len(recs) + 1)
        recs["model"] = model_name
        recommendation_rows.append(recs)

    score_results = pd.DataFrame(user_scores)
    recommendations = pd.concat(recommendation_rows, ignore_index=True)

    recommendations = recommendations.merge(
        positive_counts,
        on="movieId",
        how="left",
    )

    recommendations["global_positive_ratings"] = (
        recommendations["global_positive_ratings"].fillna(0)
    )

    return {
        "model": model_name,
        "precision_at_10": score_results["precision_at_10"].mean(),
        "recall_at_10": score_results["recall_at_10"].mean(),
        "ndcg_at_10": score_results["ndcg_at_10"].mean(),
        "rated_precision_at_10": score_results["rated_precision_at_10"].mean(),
        "users_with_rated_recommendations": score_results["rated_precision_at_10"].notna().sum(),
        "unique_recommended_movies": recommendations["movieId"].nunique(),
        "catalog_coverage": recommendations["movieId"].nunique() / movies["movieId"].nunique(),
        "avg_global_positive_ratings": recommendations["global_positive_ratings"].mean(),
        "median_global_positive_ratings": recommendations["global_positive_ratings"].median(),
    }

## Two-Signal Hybrid Weight Sweep

Compare popularity-content hybrids without a separate genre score. The content representation already includes overview and genre text.

In [3]:
popularity_content_weight_grid = [
    {"popularity_weight": 0.90, "content_weight": 0.10},
    {"popularity_weight": 0.80, "content_weight": 0.20},
    {"popularity_weight": 0.70, "content_weight": 0.30},
    {"popularity_weight": 0.60, "content_weight": 0.40},
    {"popularity_weight": 0.50, "content_weight": 0.50},
]

popularity_content_results = []

for weights in popularity_content_weight_grid:
    model_name = (
        f"pop_content_"
        f"{weights['popularity_weight']:.2f}_"
        f"{weights['content_weight']:.2f}"
    )

    metrics = evaluate_recommender(
        lambda user_id, k: recommend_popularity_content_hybrid_movies_from_vectors(
            train_ratings=train,
            movie_content=movie_content,
            content_vectors=content_vectors,
            user_id=user_id,
            k=k,
            popularity_weight=weights["popularity_weight"],
            content_weight=weights["content_weight"],
        ),
        model_name=model_name,
    )

    metrics["popularity_weight"] = weights["popularity_weight"]
    metrics["content_weight"] = weights["content_weight"]
    popularity_content_results.append(metrics)

popularity_content_results = pd.DataFrame(popularity_content_results)

popularity_content_results.sort_values(
    ["ndcg_at_10", "catalog_coverage"],
    ascending=[False, False],
)

,model,precision_at_10,recall_at_10,ndcg_at_10,rated_precision_at_10,users_with_rated_recommendations,unique_recommended_movies,catalog_coverage,avg_global_positive_ratings,median_global_positive_ratings,popularity_weight,content_weight
0,pop_content_0.90_0.10,0.054918,0.046729,0.072952,0.738574,229,95,0.009752,163.412295,155.0,0.9,0.1
1,pop_content_0.80_0.20,0.054426,0.046487,0.069349,0.742391,230,139,0.014268,157.185738,146.0,0.8,0.2
2,pop_content_0.70_0.30,0.054590,0.047314,0.065331,0.758333,236,284,0.029152,140.089508,139.0,0.7,0.3
3,pop_content_0.60_0.40,0.042459,0.034789,0.050325,0.740317,210,569,0.058407,109.490492,100.0,0.6,0.4
4,pop_content_0.50_0.50,0.033934,0.028388,0.039416,0.691453,195,766,0.078629,79.784590,60.0,0.5,0.5


## Compare Final Candidate Models

Compare the popularity baseline, the best two-signal candidates, and representative three-signal hybrids.

In [4]:
candidate_results = []

candidate_results.append(
    evaluate_recommender(
        lambda user_id, k: recommend_popular_movies(
            train_ratings=train,
            movies=movies,
            user_id=user_id,
            k=k,
        ),
        model_name="popularity_baseline",
    )
)

candidate_results.append(
    evaluate_recommender(
        lambda user_id, k: recommend_popularity_content_hybrid_movies_from_vectors(
            train_ratings=train,
            movie_content=movie_content,
            content_vectors=content_vectors,
            user_id=user_id,
            k=k,
            popularity_weight=0.80,
            content_weight=0.20,
        ),
        model_name="two_signal_accuracy_candidate",
    )
)

candidate_results.append(
    evaluate_recommender(
        lambda user_id, k: recommend_popularity_content_hybrid_movies_from_vectors(
            train_ratings=train,
            movie_content=movie_content,
            content_vectors=content_vectors,
            user_id=user_id,
            k=k,
            popularity_weight=0.60,
            content_weight=0.40,
        ),
        model_name="two_signal_discovery_candidate",
    )
)

candidate_results.append(
    evaluate_recommender(
        lambda user_id, k: recommend_true_hybrid_movies_from_vectors(
            train_ratings=train,
            movie_content=movie_content,
            content_vectors=content_vectors,
            user_id=user_id,
            k=k,
            popularity_weight=0.60,
            content_weight=0.30,
            genre_weight=0.10,
        ),
        model_name="three_signal_balanced_candidate",
    )
)

candidate_results = pd.DataFrame(candidate_results)

candidate_results.sort_values(
    ["ndcg_at_10", "catalog_coverage"],
    ascending=[False, False],
)

,model,precision_at_10,recall_at_10,ndcg_at_10,rated_precision_at_10,users_with_rated_recommendations,unique_recommended_movies,catalog_coverage,avg_global_positive_ratings,median_global_positive_ratings
0,popularity_baseline,0.057869,0.049672,0.075906,0.771876,221,91,0.009341,165.132131,155.0
1,two_signal_accuracy_candidate,0.054426,0.046487,0.069349,0.742391,230,139,0.014268,157.185738,146.0
3,three_signal_balanced_candidate,0.051967,0.043990,0.064451,0.771577,241,377,0.038698,124.553770,127.0
2,two_signal_discovery_candidate,0.042459,0.034789,0.050325,0.740317,210,569,0.058407,109.490492,100.0


## Recommendation

Use this section to choose the final offline candidate and explain the product tradeoff.

In [5]:
candidate_results[[
    "model",
    "precision_at_10",
    "recall_at_10",
    "ndcg_at_10",
    "rated_precision_at_10",
    "unique_recommended_movies",
    "catalog_coverage",
    "avg_global_positive_ratings",
]]

,model,precision_at_10,recall_at_10,ndcg_at_10,rated_precision_at_10,unique_recommended_movies,catalog_coverage,avg_global_positive_ratings
0,popularity_baseline,0.057869,0.049672,0.075906,0.771876,91,0.009341,165.132131
1,two_signal_accuracy_candidate,0.054426,0.046487,0.069349,0.742391,139,0.014268,157.185738
2,two_signal_discovery_candidate,0.042459,0.034789,0.050325,0.740317,569,0.058407,109.490492
3,three_signal_balanced_candidate,0.051967,0.043990,0.064451,0.771577,377,0.038698,124.553770
